# KTH LRCN Temporal Clipping Pipeline Replica

This notebook is a continuation of the temporal clipping workflow and represents the ready-to-use preprocessing pipeline for the KTH action recognition project.

The pipeline follows the same idea as the main notebook: sample frames from each video over time, resize them, normalize them, and prepare a sequence that can be used as input to an LRCN model.

This version is written as a more complete and reusable version of the workflow, suitable for extension into a full dataset loader.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path(r"C:\NNDL Lab Proj\KTH")
print("Project directory:", BASE_DIR)

## Step 1: Define action classes and parameters

The KTH dataset contains multiple action folders. Each folder represents one class, such as walking, running, handclapping, and handwaving. We define the class names and the clipping parameters used to build a consistent temporal representation.

In [ ]:
ACTION_CLASSES = [
    "walking",
    "running",
    "handclapping",
    "handwaving",
]

num_frames = 16
frame_step = 6
sample_indices = np.arange(0, num_frames * frame_step, frame_step)

print("Action classes:", ACTION_CLASSES)
print("Sample indices:", sample_indices)

## Step 2: Create a function to extract a temporal clip

This function loads a video, selects frames at the desired time positions, and returns the sampled frames as a NumPy array.

In [ ]:
def extract_temporal_clip(video_path, sample_indices, target_size=(224, 224)):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise FileNotFoundError(f"Unable to open video: {video_path}")

    frames = []

    for index in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ret, frame = cap.read()

        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_resized = cv2.resize(frame_rgb, target_size)
            frame_normalized = frame_resized.astype(np.float32) / 255.0
            frames.append(frame_normalized)
        else:
            print(f"Warning: frame at index {index} could not be read.")

    cap.release()

    if not frames:
        raise ValueError(f"No frames extracted from: {video_path}")

    clip = np.stack(frames, axis=0)
    return clip

In [ ]:
video_path = BASE_DIR / "walking" / "person01_walking_d1_uncomp.avi"
clip = extract_temporal_clip(video_path, sample_indices)

print("Clip shape:", clip.shape)
print("Value range:", clip.min(), clip.max())

## Step 3: Visualize the extracted temporal clip

The sequence below shows the frames selected from the video. This helps confirm that the sampled frames preserve the motion pattern of the action.

In [ ]:
plt.figure(figsize=(12, 8))

for i, frame in enumerate(clip):
    plt.subplot(4, 4, i + 1)
    plt.imshow(frame)
    plt.title(f"Frame {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 4: Build the dataset structure

To convert this into a complete pipeline, we create a list of videos grouped by action class and prepare each one into a clip with a label.

In [ ]:
def build_dataset(base_dir, action_classes, sample_indices):
    dataset = []

    for class_name in action_classes:
        class_dir = base_dir / class_name

        if not class_dir.exists():
            print(f"Class folder not found: {class_dir}")
            continue

        for video_file in sorted(class_dir.glob("*.avi")):
            clip = extract_temporal_clip(video_file, sample_indices)
            dataset.append((clip, class_name))

    return dataset

# Example: build only a small dataset for testing
small_dataset = build_dataset(BASE_DIR, ACTION_CLASSES[:1], sample_indices)
print("Dataset length:", len(small_dataset))
print("Example label:", small_dataset[0][1])
print("Example clip shape:", small_dataset[0][0].shape)

## Why this is the ready pipeline

This workflow is now in a pipeline-ready form because it provides the core steps needed for model training:

- access to the dataset folders
- action class mapping
- temporal frame sampling
- frame decoding and resizing
- normalization to a consistent range
- clip construction for each video
- dataset preparation for training and evaluation

This is the essential preprocessing stage for LRCN activity recognition.

## Final note

The temporal clipping process is the foundation of the KTH LRCN pipeline. Once all videos are processed in this way, the data can be converted into tensors, split into train/validation/test sets, and passed into a recurrent network for action classification.